# Malayalam LLM Evaluation: A Comparative Study of Three Language Models

## Project Overview

This project investigates the performance of three open-source Large Language Models (LLMs) on Malayalam reading comprehension.

The study compares Qwen 2.5 3B Instruct, Gemma 2 2B IT, and Llama 3.2 3B Instruct using the Belebele multilingual reading comprehension benchmark.

## Research Objectives

- Evaluate the Malayalam reading comprehension capabilities of three LLMs.
- Compare English and Malayalam zero-shot prompting.
- Investigate the impact of few-shot prompting.
- Analyze tokenizer efficiency across Malayalam and English.
- Identify model weaknesses through incorrect prediction analysis.

## Experimental Setup

- Platform: Google Colab
- GPU: NVIDIA Tesla T4
- Dataset: Belebele
- Malayalam evaluation questions: 200
- Models: 3
- Prompting conditions: 3
- Total recorded predictions: 1,800

## Models Evaluated

1. Qwen 2.5 3B Instruct
2. Gemma 2 2B IT
3. Llama 3.2 3B Instruct

## Research Focus

Understanding how effectively modern language models process Malayalam compared with English, and investigating how prompting strategies influence their performance.

## 1. GPU Environment Verification

Before loading large language models, we need to verify whether the Google Colab environment provides GPU acceleration.

This cell:
- Checks CUDA availability.
- Identifies the GPU model.
- Confirms whether the environment is ready for model inference.

GPU acceleration helps reduce the time required to load and evaluate large language models.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Go to Runtime > Change runtime type > T4 GPU, Save, then rerun from here.")

GPU available: True
GPU name: Tesla T4


## 2. Installing Dependencies and Authentication

This cell installs the essential Python libraries required for the experiment.

Libraries used:
- Transformers: Loading and running pretrained language models.
- Accelerate: Efficient model execution and device management.
- Datasets: Loading the Belebele benchmark dataset.
- SentencePiece: Tokenization support.
- Hugging Face Hub: Accessing pretrained models.

Hugging Face authentication is performed to access models that may require user authorization.

In [ ]:
!pip install -q -U transformers accelerate datasets sentencepiece huggingface_hub
from huggingface_hub import login
login()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


## 3. Experiment Configuration

This cell defines the main settings and configurations for the evaluation.

Key configurations:
- Random seed: 42 for reproducibility.
- Evaluation samples: 200 questions.
- Few-shot examples: 3 demonstrations.
- Models: Llama 3.2 3B Instruct for this model-running session.

Three evaluation conditions are defined:

1. English Zero-Shot: English instructions without examples.
2. Malayalam Zero-Shot: Malayalam instructions without examples.
3. English Few-Shot: English instructions with three examples.

The answer format is restricted to four multiple-choice options: A, B, C, and D.

These configurations ensure consistent evaluation conditions.

In [ ]:
import json, random, gc, time
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
N_EVAL = 200
N_SHOT = 3

MODELS = {

    "llama": "meta-llama/Llama-3.2-3B-Instruct",
}

LETTERS = ["A", "B", "C", "D"]
INSTR = {
    "en": "Read the passage and answer the multiple-choice question. Reply with only one letter: A, B, C, or D.",
    "ml": "ഇനിപ്പറയുന്ന ഭാഗം വായിച്ച് ചോദ്യത്തിന് ശരിയായ ഉത്തരം തിരഞ്ഞെടുക്കുക. A, B, C, D എന്നിവയിൽ ഒരു അക്ഷരം മാത്രം ഉത്തരമായി നൽകുക.",
}
CONDITIONS = [("en_zero", "en", False), ("ml_zero", "ml", False), ("en_few", "en", True)]
print("Settings ready. N_EVAL =", N_EVAL, "| models:", list(MODELS))

Settings ready. N_EVAL = 200 | models: ['llama']


## 4. Loading and Preparing the Belebele Dataset

This cell loads the Malayalam configuration of the Belebele benchmark dataset.

Dataset:
- Source: Facebook Belebele.
- Language: Malayalam (mal_Mlym).
- Split: Test.

A fixed random seed is used to shuffle the dataset and select:
- 200 questions for model evaluation.
- 3 separate examples for few-shot prompting.

Separating evaluation questions from demonstration examples helps prevent data leakage between the examples and the evaluation sample.

In [ ]:
try:
    ds = load_dataset("facebook/belebele", "mal_Mlym", split="test")
except Exception as e:
    print("First attempt failed:", e)
    ds = load_dataset("facebook/belebele", "mal_Mlym", split="test", trust_remote_code=True)

print("Number of questions:", len(ds))
print("Columns:", ds.column_names)

idx = list(range(len(ds)))
random.Random(SEED).shuffle(idx)
eval_idx = idx[:N_EVAL]
shot_idx = idx[N_EVAL:N_EVAL + N_SHOT]
print("Eval questions:", len(eval_idx), "| few-shot examples:", len(shot_idx))

Number of questions: 900
Columns: ['link', 'question_number', 'flores_passage', 'question', 'mc_answer1', 'mc_answer2', 'mc_answer3', 'mc_answer4', 'correct_answer_num', 'dialect', 'ds']
Eval questions: 200 | few-shot examples: 3


## 5. Prompt Engineering and Prediction Functions

This cell defines the functions required to prepare prompts and obtain model predictions.

Main functions:

1. fmt(): Formats the passage, question, and four answer choices.
2. gold(): Extracts the correct answer label from the dataset.
3. build_messages(): Creates English or Malayalam instructions and optionally includes few-shot examples.
4. predict(): Passes the formatted prompt to the language model and obtains the prediction.

The model uses its next-token logits to compare the scores of the four possible answer letters (A, B, C, and D).

The function also records:
- Predicted answer.
- Answer format compliance.
- Number of prompt tokens.

These functions form the core of the evaluation pipeline.

In [ ]:
def fmt(ex):
    opts = "\n".join(f"{l}. {ex[f'mc_answer{i+1}']}" for i, l in enumerate(LETTERS))
    return f"Passage:\n{ex['flores_passage']}\n\nQuestion: {ex['question']}\n{opts}"

def gold(ex):
    return int(ex["correct_answer_num"]) - 1

def build_messages(ex, lang, use_shots):
    msgs = []
    if use_shots:
        for i in shot_idx:
            s = ds[i]
            msgs.append({"role": "user", "content": INSTR[lang] + "\n\n" + fmt(s)})
            msgs.append({"role": "assistant", "content": LETTERS[gold(s)]})
    msgs.append({"role": "user", "content": INSTR[lang] + "\n\n" + fmt(ex)})
    return msgs

@torch.no_grad()
def predict(model, tok, msgs, letter_ids):
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    try:
        logits = model(**enc, logits_to_keep=1).logits[0, -1]
    except TypeError:
        logits = model(**enc).logits[0, -1]
    pred = int(logits[letter_ids].argmax())
    follows_format = int(int(logits.argmax()) in letter_ids)
    return pred, follows_format, int(enc["input_ids"].shape[1])

print("Functions ready.")

Functions ready.


## 6. Running the Language Model Evaluation

This cell performs the actual inference process using the configured language model.

Workflow:

1. Load the pretrained tokenizer and language model.
2. Move the model to the available GPU.
3. Evaluate all 200 questions under the three prompting conditions.
4. Record the predicted answer and correct answer.
5. Calculate whether the prediction is correct.
6. Record answer format compliance and prompt token count.
7. Save the predictions in JSONL format.

The results are saved separately using model-specific filenames.

For this session, the configured model is Llama 3.2 3B Instruct.

GPU memory is cleared after evaluation to support efficient resource management.

In [ ]:
assert torch.cuda.is_available(), "No GPU! Fix the runtime first (Cell 0)."

results = []
t0 = time.time()
for mname, mid in MODELS.items():
    print(f"Loading {mname}: {mid}")
    tok = AutoTokenizer.from_pretrained(mid)
    model = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16, device_map={"": 0})
    model.eval()
    print("Loaded. GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))
    letter_ids = [tok.encode(l, add_special_tokens=False)[0] for l in LETTERS]
    for cname, lang, shots in CONDITIONS:
        for n, i in enumerate(eval_idx):
            ex = ds[i]
            pred, fmt_ok, n_tok = predict(model, tok, build_messages(ex, lang, shots), letter_ids)
            results.append(dict(model=mname, cond=cname, idx=i, pred=pred, gold=gold(ex),
                                correct=int(pred == gold(ex)), follows_format=fmt_ok,
                                prompt_tokens=n_tok))
            if n % 10 == 0:
                print(f"{mname} {cname} {n}/{len(eval_idx)}  elapsed {time.time()-t0:.0f}s")
        with open(f"results_{mname}.jsonl", "w") as f:
            for r in results:
                if r["model"] == mname:
                    f.write(json.dumps(r) + "\n")
    del model
    gc.collect(); torch.cuda.empty_cache()
print("DONE", len(results), "answers in", round(time.time() - t0), "seconds")

Loading llama: meta-llama/Llama-3.2-3B-Instruct


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded. GPU memory used (GB): 6.4
llama en_zero 0/200  elapsed 146s
llama en_zero 10/200  elapsed 151s
llama en_zero 20/200  elapsed 154s
llama en_zero 30/200  elapsed 158s
llama en_zero 40/200  elapsed 162s
llama en_zero 50/200  elapsed 165s
llama en_zero 60/200  elapsed 169s
llama en_zero 70/200  elapsed 173s
llama en_zero 80/200  elapsed 178s
llama en_zero 90/200  elapsed 181s
llama en_zero 100/200  elapsed 185s
llama en_zero 110/200  elapsed 189s
llama en_zero 120/200  elapsed 194s
llama en_zero 130/200  elapsed 198s
llama en_zero 140/200  elapsed 203s
llama en_zero 150/200  elapsed 207s
llama en_zero 160/200  elapsed 211s
llama en_zero 170/200  elapsed 215s
llama en_zero 180/200  elapsed 219s
llama en_zero 190/200  elapsed 224s
llama ml_zero 0/200  elapsed 228s
llama ml_zero 10/200  elapsed 234s
llama ml_zero 20/200  elapsed 238s
llama ml_zero 30/200  elapsed 243s
llama ml_zero 40/200  elapsed 248s
llama ml_zero 50/200  elapsed 252s
llama ml_zero 60/200  elapsed 257s
llama ml_zero

## 7. Loading Model Prediction Results

This cell loads the previously saved JSONL result files.

Since the three models were evaluated across separate sessions, their prediction files are collected and combined into a single results list.

Each record contains:
- Model name.
- Evaluation condition.
- Question index.
- Predicted answer.
- Correct answer.
- Prediction correctness.
- Answer format compliance.
- Prompt token count.

Combining these results allows us to perform a comparative analysis of all three models.

In [ ]:
import glob
results = []
for fn in sorted(glob.glob("results_*.jsonl")):
    with open(fn) as f:
        results += [json.loads(line) for line in f]
print(len(results), "answers loaded from", sorted(glob.glob("results_*.jsonl")))

1800 answers loaded from ['results_gemma.jsonl', 'results_llama.jsonl', 'results_qwen.jsonl']


## 8. Model Accuracy and Statistical Confidence Analysis

This cell calculates the performance of each model under the three evaluation conditions.

Metrics:

1. Accuracy: Measures the percentage of correctly answered questions.
2. 95% Confidence Interval: Estimates the uncertainty around the observed accuracy using bootstrap resampling.
3. Letter Rate: Measures how often the model's highest-scoring next token belongs to the allowed answer choices (A, B, C, or D).

The bootstrap method uses 2,000 resampling iterations to estimate the confidence intervals.

For a four-option multiple-choice task, random guessing has an expected accuracy of 25%.

This analysis helps compare model performance beyond accuracy alone.

In [ ]:
def bootstrap_ci(correct, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    c = np.array(correct)
    means = [rng.choice(c, len(c)).mean() for _ in range(n)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    return c.mean(), lo, hi

print(f"{'model':8}{'condition':10}{'acc':>7}{'95% CI':>18}{'letter-rate':>13}")
for m in sorted(set(r["model"] for r in results)):
    for cname, _, _ in CONDITIONS:
        rows = [r for r in results if r["model"] == m and r["cond"] == cname]
        acc, lo, hi = bootstrap_ci([r["correct"] for r in rows])
        fr = np.mean([r["follows_format"] for r in rows])
        print(f"{m:8}{cname:10}{acc:7.3f}   [{lo:.3f}, {hi:.3f}]{fr:13.2f}")
print("Chance level = 0.25")

model   condition     acc            95% CI  letter-rate
gemma   en_zero     0.475   [0.405, 0.545]         1.00
gemma   ml_zero     0.210   [0.155, 0.265]         0.00
gemma   en_few      0.495   [0.430, 0.565]         0.99
llama   en_zero     0.395   [0.330, 0.465]         1.00
llama   ml_zero     0.460   [0.395, 0.530]         1.00
llama   en_few      0.520   [0.450, 0.590]         1.00
qwen    en_zero     0.400   [0.335, 0.465]         1.00
qwen    ml_zero     0.335   [0.270, 0.400]         0.12
qwen    en_few      0.465   [0.400, 0.535]         1.00
Chance level = 0.25


## 9. Comparing Tokenization Efficiency Across Languages

This cell investigates how efficiently different model tokenizers represent Malayalam and English text.

Three tokenizers are evaluated:
- Qwen 2.5 3B Instruct.
- Gemma 2 2B IT.
- Llama 3.2 3B Instruct.

The analysis calculates the average number of tokens per whitespace-separated word.

The same comparison is performed using Malayalam and English passages from the Belebele dataset.

A higher tokens-per-word value indicates that more tokens are required to represent the text.

This experiment helps investigate differences in tokenization efficiency between Malayalam and English.

In [ ]:
ALL_MODELS = {
    "qwen": "Qwen/Qwen2.5-3B-Instruct",
    "gemma": "google/gemma-2-2b-it",
    "llama": "meta-llama/Llama-3.2-3B-Instruct",
}
ds_en = load_dataset("facebook/belebele", "eng_Latn", split="test")
ml_texts = list(dict.fromkeys(ds["flores_passage"]))[:200]
en_texts = list(dict.fromkeys(ds_en["flores_passage"]))[:200]

def fertility(tok, texts):
    words = sum(len(t.split()) for t in texts)
    toks = sum(len(tok.encode(t, add_special_tokens=False)) for t in texts)
    return toks / words

for mname, mid in ALL_MODELS.items():
    try:
        tok = AutoTokenizer.from_pretrained(mid)
        print(f"{mname}: Malayalam {fertility(tok, ml_texts):.2f} tokens/word, "
              f"English {fertility(tok, en_texts):.2f} tokens/word")
    except Exception as e:
        print(f"{mname}: skipped ({type(e).__name__})")

qwen: Malayalam 13.45 tokens/word, English 1.25 tokens/word
gemma: Malayalam 5.86 tokens/word, English 1.23 tokens/word
llama: Malayalam 16.30 tokens/word, English 1.23 tokens/word


## 10. Extracting Incorrect Predictions for Error Analysis

This cell identifies the questions that each model answered incorrectly.

The incorrect predictions are:
- Extracted from the combined results.
- Randomly shuffled using a fixed seed.
- Limited to a maximum of 60 examples.
- Combined with their original questions, answer choices, and passages.

The selected records are exported into a JSONL file named wrong_sample.jsonl.

This dataset will be used for manual inspection to understand the types of mistakes made by the models.

In [ ]:
wrong = [r for r in results if not r["correct"]]
random.Random(SEED).shuffle(wrong)
with open("wrong_sample.jsonl", "w") as f:
    for r in wrong[:60]:
        ex = ds[r["idx"]]
        f.write(json.dumps({**r, "question": ex["question"],
                            "options": [ex[f"mc_answer{i+1}"] for i in range(4)],
                            "passage": ex["flores_passage"]},
                           ensure_ascii=False) + "\n")
print("saved", min(60, len(wrong)), "of", len(wrong), "wrong answers")

saved 60 of 1049 wrong answers


## 11. Manual Inspection of Model Errors

This cell displays selected incorrect predictions for detailed examination.

For each example, it displays:
- Model name.
- Prompting condition.
- Original passage.
- Question.
- Four answer choices.
- Correct answer.
- Model prediction.

The batch size and passage length can be adjusted to inspect different examples.

Manual inspection helps identify potential weaknesses in Malayalam reading comprehension, question understanding, and answer selection.

In [ ]:
START = 0
BATCH = 10
PASSAGE_CHARS = 700

with open("wrong_sample.jsonl", encoding="utf-8") as f:
    rows = [json.loads(l) for l in f]

for k, r in enumerate(rows[START:START + BATCH], start=START):
    print("=" * 80)
    print(f"#{k} | model={r['model']} | condition={r['cond']} | model wanted a letter={bool(r['follows_format'])}")
    print("PASSAGE:", r["passage"][:PASSAGE_CHARS], "...")
    print("QUESTION:", r["question"])
    for L, o in zip("ABCD", r["options"]):
        print(f"  {L}. {o}")
    print("CORRECT:", "ABCD"[r["gold"]], "| MODEL PICKED:", "ABCD"[r["pred"]])

#0 | model=gemma | condition=ml_zero | model wanted a letter=False
PASSAGE: "മുതിർന്ന രണ്ട് കുട്ടികളുള്ള വിവാഹം കഴിഞ്ഞ ഡുവാൽ, ഈ കഥ ബന്ധപ്പെട്ടിരിക്കുന്ന മില്ലറിൽ വലിയ മതിപ്പുണ്ടാക്കിയില്ല. ""അഭിപ്രായം ആരാഞ്ഞപ്പോൾ മില്ലർ പറഞ്ഞു, """"ഹിയറിംഗിനിടെ മൈക്ക് വളരെയധികം സംസാരിക്കുന്നു...ഞാൻ തയ്യാറെടുത്തിരുന്നതിനാൽ അദ്ദേഹം പറയുന്നത് കേൾക്കുന്നുണ്ടായിരുന്നില്ല.""""""" ...
QUESTION: ഖണ്ഡിക അനുസരിച്ച്, മില്ലറുടെ അനുഭവത്തിൽ ഹിയറിംഗിൽ ഇനിപ്പറയുന്നവയിൽ ശരിയായത് ഏതാണ്?
  A. ഹിയറിംഗിനിടെ ഡുവാൽ മില്ലറിൽ ഒരു മതിപ്പ് ഉണ്ടാക്കി
  B. അവൻ തയ്യാറെടുക്കുമ്പോൾ അത് ശ്രദ്ധിച്ചില്ല
  C. ഹിയറിങ്ങിനിടയിൽ ഡുവാലിന്റെ രണ്ട് കുട്ടികളും ഒരുപാട് സംസാരിച്ചു
  D. ഡുവാലിന്റെ കഥ അവൻ നന്നായി ശ്രദ്ധിച്ചുകൊണ്ടിരുന്നു
CORRECT: B | MODEL PICKED: A
#1 | model=qwen | condition=ml_zero | model wanted a letter=False
PASSAGE: """കത്രീന ചുഴലിക്കാറ്റിനെ തുടർന്ന് ദുരിതാശ്വാസത്തിനും പുനർനിർമാണത്തിനും വേണ്ടിയുള്ള ചെലവഴിക്കൽ സംബന്ധിച്ച വിവാദമാണ് ചർച്ചയ്ക്ക് വഴിതുറന്നത്; ചില യാഥാസ്ഥിതിക സാമ്പത്തിക വാദികൾ """"ബുഷിന്റെ ന്യൂ ഓർലിയൻസ് ഡീൽ"""" എന്ന

## 12. Distribution of Predicted Answer Choices

This cell examines how frequently each model selects the answer choices A, B, C, and D.

The analysis is performed separately for every model and prompting condition.

The purpose is to identify possible answer-position preferences or unusual prediction patterns.

For example, a model that frequently selects option A may have an answer-position bias.

This analysis provides additional insight into model prediction behaviour beyond overall accuracy.

In [ ]:
from collections import Counter
for m in sorted(set(r["model"] for r in results)):
    for cname, _, _ in CONDITIONS:
        rows = [r for r in results if r["model"] == m and r["cond"] == cname]
        counts = Counter("ABCD"[r["pred"]] for r in rows)
        print(m, cname, dict(sorted(counts.items())))

gemma en_zero {'A': 36, 'B': 39, 'C': 55, 'D': 70}
gemma ml_zero {'A': 199, 'D': 1}
gemma en_few {'A': 41, 'B': 61, 'C': 60, 'D': 38}
llama en_zero {'A': 8, 'B': 21, 'C': 149, 'D': 22}
llama ml_zero {'A': 39, 'B': 32, 'C': 92, 'D': 37}
llama en_few {'A': 37, 'B': 76, 'C': 41, 'D': 46}
qwen en_zero {'A': 57, 'B': 62, 'C': 61, 'D': 20}
qwen ml_zero {'A': 2, 'B': 35, 'C': 99, 'D': 64}
qwen en_few {'A': 65, 'B': 63, 'C': 44, 'D': 28}


## 13. Statistical Comparison of Prompting Strategies

This cell compares the performance differences between the three prompting conditions for each model.

Comparisons include:

- English Few-Shot vs English Zero-Shot.
- English Zero-Shot vs Malayalam Zero-Shot.
- English Few-Shot vs Malayalam Zero-Shot.

A paired bootstrap method with 5,000 iterations is used to estimate the mean accuracy difference and its 95% confidence interval.

Because the same evaluation questions are used across conditions, paired comparisons help control for differences in question difficulty.

This analysis investigates how language choice and few-shot examples influence model performance.

In [ ]:
def paired_diff(m, a, b, n=5000, seed=0):
    da = {r["idx"]: r["correct"] for r in results if r["model"] == m and r["cond"] == a}
    db = {r["idx"]: r["correct"] for r in results if r["model"] == m and r["cond"] == b}
    keys = sorted(da)
    d = np.array([da[k] - db[k] for k in keys])
    rng = np.random.default_rng(seed)
    boots = [rng.choice(d, len(d)).mean() for _ in range(n)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return d.mean(), lo, hi

for m in sorted(set(r["model"] for r in results)):
    for a, b in [("en_few", "en_zero"), ("en_zero", "ml_zero"), ("en_few", "ml_zero")]:
        mean, lo, hi = paired_diff(m, a, b)
        print(f"{m}: {a} minus {b}: {mean:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")

gemma: en_few minus en_zero: +0.020   95% CI [-0.050, +0.090]
gemma: en_zero minus ml_zero: +0.265   95% CI [+0.175, +0.355]
gemma: en_few minus ml_zero: +0.285   95% CI [+0.190, +0.375]
llama: en_few minus en_zero: +0.125   95% CI [+0.045, +0.205]
llama: en_zero minus ml_zero: -0.065   95% CI [-0.130, -0.005]
llama: en_few minus ml_zero: +0.060   95% CI [-0.005, +0.125]
qwen: en_few minus en_zero: +0.065   95% CI [+0.010, +0.120]
qwen: en_zero minus ml_zero: +0.065   95% CI [-0.010, +0.140]
qwen: en_few minus ml_zero: +0.130   95% CI [+0.045, +0.215]


## 14. Comparative Analysis Between Language Models

This cell compares the performance of different models under identical prompting conditions.

Models compared:
- Qwen 2.5 3B Instruct.
- Gemma 2 2B IT.
- Llama 3.2 3B Instruct.

The comparisons are performed under:
- English Zero-Shot.
- English Few-Shot.

A paired bootstrap method with 5,000 iterations estimates the accuracy difference and its 95% confidence interval.

Using the same question indices ensures that the models are compared on matching evaluation examples.

This helps identify which model performs better under the same experimental conditions.

In [ ]:
from itertools import combinations
def model_diff(m1, m2, cond, n=5000, seed=0):
    d1 = {r["idx"]: r["correct"] for r in results if r["model"] == m1 and r["cond"] == cond}
    d2 = {r["idx"]: r["correct"] for r in results if r["model"] == m2 and r["cond"] == cond}
    keys = sorted(set(d1) & set(d2))
    d = np.array([d1[k] - d2[k] for k in keys])
    rng = np.random.default_rng(seed)
    boots = [rng.choice(d, len(d)).mean() for _ in range(n)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return d.mean(), lo, hi

models = sorted(set(r["model"] for r in results))
for m1, m2 in combinations(models, 2):
    for cond in ["en_zero", "en_few"]:
        mean, lo, hi = model_diff(m1, m2, cond)
        print(f"{cond}: {m1} minus {m2}: {mean:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")

en_zero: gemma minus llama: +0.080   95% CI [-0.005, +0.170]
en_few: gemma minus llama: -0.025   95% CI [-0.105, +0.055]
en_zero: gemma minus qwen: +0.075   95% CI [-0.010, +0.160]
en_few: gemma minus qwen: +0.030   95% CI [-0.060, +0.120]
en_zero: llama minus qwen: -0.005   95% CI [-0.090, +0.080]
en_few: llama minus qwen: +0.055   95% CI [-0.030, +0.140]


## 16. Uploading Previously Generated Model Results

This cell provides a file upload interface in Google Colab.

It allows previously generated model prediction files to be uploaded into the current runtime.

This is useful because the models were evaluated in separate sessions, and their JSONL result files need to be combined for comparative analysis.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving results_qwen (1).jsonl to results_qwen (1).jsonl


## 17. Reinitializing the Analysis Environment

This cell restores the dependencies and configurations required for the final analysis.

It performs the following operations:

1. Imports the required Python libraries.
2. Defines the random seed and answer labels.
3. Restores the three evaluation conditions.
4. Reloads the Malayalam Belebele dataset.
5. Reads the available JSONL prediction files.
6. Defines the bootstrap confidence interval function.

This allows the analysis to continue in a fresh Google Colab session without repeating the entire model inference process.

In [ ]:
import json, random, glob, csv
import numpy as np
from collections import Counter
from datasets import load_dataset

SEED = 42
LETTERS = ["A", "B", "C", "D"]
CONDITIONS = [("en_zero", "en", False), ("ml_zero", "ml", False), ("en_few", "en", True)]

try:
    ds = load_dataset("facebook/belebele", "mal_Mlym", split="test")
except Exception:
    ds = load_dataset("facebook/belebele", "mal_Mlym", split="test", trust_remote_code=True)
print("Dataset questions:", len(ds))

results = []
for fn in sorted(glob.glob("results_*.jsonl")):
    with open(fn) as f:
        results += [json.loads(line) for line in f]
print(len(results), "answers loaded from", sorted(glob.glob("results_*.jsonl")))

def bootstrap_ci(correct, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    c = np.array(correct)
    means = [rng.choice(c, len(c)).mean() for _ in range(n)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    return c.mean(), lo, hi

README.md:   0%|          | 0.00/25.5k [00:00<?, ?B/s]

mal_Mlym.jsonl:   0%|          | 0.00/3.83M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

Dataset questions: 900
600 answers loaded from ['results_qwen (1).jsonl']


## 18. Uploading Additional Model Prediction Files

This cell provides another file upload interface to import prediction files generated during separate model evaluation sessions.

The uploaded files are required to combine the results of different language models for the final comparative analysis.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving results_gemma (1).jsonl to results_gemma (1).jsonl


## 19. Combining Available Model Predictions

This cell reads all available results_*.jsonl files and combines their prediction records.

It also displays:
- Total number of loaded predictions.
- Names of the loaded result files.
- Number of predictions generated by each model.

This helps verify that the model results have been successfully imported before proceeding with further analysis.

In [ ]:
results = []
for fn in sorted(glob.glob("results_*.jsonl")):
    with open(fn) as f:
        results += [json.loads(line) for line in f]
print(len(results), "answers loaded from", sorted(glob.glob("results_*.jsonl")))
print(Counter(r["model"] for r in results))

1200 answers loaded from ['results_gemma (1).jsonl', 'results_qwen (1).jsonl']
Counter({'gemma': 600, 'qwen': 600})


## 20. Uploading the Remaining Model Results

This cell provides the final file upload step for importing any remaining model prediction files.

It ensures that the results from all three evaluated language models are available in the Colab runtime before performing the final comparison.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving results_llama.jsonl to results_llama.jsonl


## 21. Verifying the Combined Evaluation Dataset

This cell reloads all available JSONL prediction files and combines them into a single results list.

It calculates the total number of prediction records and displays the number of records associated with each model.

The expected total is:

200 questions × 3 prompting conditions × 3 models = 1,800 predictions.

This verification helps ensure that all three models and their evaluation conditions are represented in the final analysis.

In [ ]:
results = []
for fn in sorted(glob.glob("results_*.jsonl")):
    with open(fn) as f:
        results += [json.loads(line) for line in f]
print(len(results), "answers loaded from", sorted(glob.glob("results_*.jsonl")))
print(Counter(r["model"] for r in results))

1800 answers loaded from ['results_gemma (1).jsonl', 'results_llama.jsonl', 'results_qwen (1).jsonl']
Counter({'gemma': 600, 'llama': 600, 'qwen': 600})


## 22. Checking the Distribution of Correct Answer Positions

This cell examines the distribution of the correct answer choices (A, B, C, and D) in the evaluation sample.

It counts how many questions have each answer letter as the correct option.

This check helps identify possible imbalances in the answer-position distribution of the selected questions.

Understanding this distribution is useful when interpreting model prediction patterns and potential answer-position biases.

In [ ]:
first_model = results[0]["model"]
gold_counts = Counter("ABCD"[r["gold"]] for r in results
                      if r["model"] == first_model and r["cond"] == "en_zero")
print("Correct-answer letters in the 200-question sample:", dict(sorted(gold_counts.items())))

Correct-answer letters in the 200-question sample: {'A': 41, 'B': 63, 'C': 52, 'D': 44}


## 23. Exporting the Final Model Performance Summary

This cell generates the final performance summary for all three language models.

For each model and prompting condition, it calculates:

- Number of evaluated questions.
- Accuracy.
- 95% bootstrap confidence interval.
- Answer format compliance rate.

The results are saved into a CSV file named summary_table.csv.

The file is then downloaded from Google Colab for further analysis, visualization, and documentation.

This CSV serves as the main quantitative summary of the experiment.

In [ ]:
rows_out = []
for m in sorted(set(r["model"] for r in results)):
    for cname, _, _ in CONDITIONS:
        rows = [r for r in results if r["model"] == m and r["cond"] == cname]
        acc, lo, hi = bootstrap_ci([r["correct"] for r in rows])
        fr = np.mean([r["follows_format"] for r in rows])
        rows_out.append([m, cname, len(rows), round(acc, 3), round(lo, 3), round(hi, 3), round(fr, 2)])
with open("summary_table.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["model", "condition", "n", "accuracy", "ci_low", "ci_high", "letter_rate"])
    w.writerows(rows_out)
from google.colab import files
files.download("summary_table.csv")
print("done")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

done


## 24. Preparing the Final Error Analysis Dataset

This cell creates a balanced sample of incorrect predictions from the evaluated language models.

The process:
1. Identifies incorrect predictions for each model.
2. Selects up to 20 incorrect examples per model.
3. Focuses on English Zero-Shot and English Few-Shot conditions.
4. Randomly shuffles the selected examples.
5. Adds the original passage, question, and answer choices.
6. Exports the final dataset as wrong_sample.jsonl.

This file supports further manual investigation of model weaknesses and differences in reasoning or reading comprehension.

In [ ]:
wrong = []
for m in sorted(set(r["model"] for r in results)):
    w = [r for r in results if r["model"] == m and (not r["correct"]) and r["cond"] in ("en_zero", "en_few")]
    random.Random(SEED).shuffle(w)
    wrong += w[:20]
random.Random(SEED).shuffle(wrong)
with open("wrong_sample.jsonl", "w") as f:
    for r in wrong:
        ex = ds[r["idx"]]
        f.write(json.dumps({**r, "question": ex["question"],
                            "options": [ex[f"mc_answer{i+1}"] for i in range(4)],
                            "passage": ex["flores_passage"]},
                           ensure_ascii=False) + "\n")
print("saved", len(wrong), "wrong answers")
from google.colab import files
files.download("wrong_sample.jsonl")

saved 60 wrong answers


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Limitations

- The evaluation uses a sample of 200 Malayalam questions rather than the complete benchmark.
- Only three language models are included in the comparison.
- The experiment focuses on multiple-choice reading comprehension.
- Model performance may vary depending on prompting strategy and tokenization.
- The results may not represent overall Malayalam language understanding across different domains.
- Further evaluation using larger datasets and additional Indian languages is required.

# Conclusion

This project presents a comparative evaluation of three open-source language models: Qwen 2.5 3B Instruct, Gemma 2 2B IT, and Llama 3.2 3B Instruct.

Using the Malayalam Belebele benchmark, the experiment investigates how different prompting strategies affect model performance in Malayalam reading comprehension.

The study includes:
- English and Malayalam zero-shot evaluation.
- English few-shot evaluation.
- Accuracy and statistical confidence analysis.
- Malayalam and English tokenizer efficiency comparison.
- Answer distribution analysis.
- Model performance comparison.
- Manual error analysis.

The findings provide insights into the capabilities and limitations of different language models when handling Malayalam reading comprehension tasks.

The experiment also highlights the importance of evaluating large language models across Indian languages rather than relying exclusively on English-language benchmarks.

**Future Work:** Extend the evaluation to additional Indian languages, larger datasets, and more open-source language models.